In [1]:
#channel attention

import torch
import os
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torchvision.models import vit_b_16
from torch.utils.data import DataLoader,random_split

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
from torch.cuda import manual_seed
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/Breast_Cancer/dataset_cancer_v1/classificacao_binaria/40X'


transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(path,transform = transform)

train_size = int(0.8*len(full_dataset))
val_size = int(0.1*len(full_dataset))
test_size = len(full_dataset) - train_size - val_size


train_dataset,val_dataset,test_dataset = random_split(
    full_dataset,
    [train_size,val_size,test_size],
    generator= torch.Generator().manual_seed(42)
)


Mounted at /content/drive


In [4]:
class ChannelAttention(nn.Module):
  def __init__(self,in_channels,reduction=16):
    super().__init__()

    self.avg_pool = nn.AdaptiveAvgPool2d(1)
    self.max_pool = nn.AdaptiveMaxPool2d(1)

    self.fc = nn.Sequential(
        nn.Conv2d(in_channels,in_channels//reduction,1),
        nn.ReLU(),
        nn.Conv2d(in_channels//reduction,in_channels,1)
    )

    self.sigmoid = nn.Sigmoid()

  def forward(self,x):

    avg_out = self.fc(self.avg_pool(x))

    max_out = self.fc(self.max_pool(x))

    out = avg_out + max_out

    return self.sigmoid(out)



In [5]:
class ViTWithchannelAttention(nn.Module):

  def __init__(self,num_classes=2):
    super().__init__()

    self.channel_attention = ChannelAttention(in_channels=3,reduction=1)

    self.vit = vit_b_16(weights="IMAGENET1K_V1")

    self.vit.heads.head = nn.Linear(
        self.vit.heads.head.in_features,
        num_classes

    )

  def forward(self,x):

    attention = self.channel_attention(x)

    x = x*attention

    x = self.vit(x)

    return x
model = ViTWithchannelAttention().to(device)


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 199MB/s]


In [6]:
#Data Loader

train_loader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size = 32,
    shuffle = False
)

test_loader = DataLoader(
    test_dataset,
    batch_size = 32,
    shuffle = False
)

In [7]:
#Loss
import torch.optim

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr = 0.0001)

In [8]:
#training

epochs = 10
corrected = 0
totaL = 0
best_loss = float("inf")

for epoch in range(epochs):

  running_loss = 0

  model.train()

  for images, lables in train_loader:

    images = images.to(device)
    lables = lables.to(device)

    optimizer.zero_grad()

    outputs = model(images)

    loss = criterion(outputs,lables)

    loss.backward()

    optimizer.step()

    running_loss +=loss.item()

  training_loss = running_loss / len(train_loader)

  model.eval()

  val_loss = 0

  with torch.no_grad():

    for images,labels in val_loader:

      images = images.to(device)
      labels = labels.to(device)

      outputs = model(images)

      loss = criterion(outputs,labels)

      val_loss += loss.item()
      _, predicated = torch.max(outputs,1)
      corrected+= (predicated == labels).sum().item()
      totaL+=labels.size(0)

    valdation_loss = val_loss / len(val_loader)
    val_accy = corrected / totaL *100

    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {training_loss:.4f}, Validation Loss: {valdation_loss:.4f},Validation acc: {val_accy:.2f}")

    if valdation_loss < best_loss:
      best_loss = valdation_loss
      torch.save(model.state_dict(),"best_model.pth")


Epoch 1/10, Training Loss: 0.5938, Validation Loss: 0.4885,Validation acc: 80.90
Epoch 2/10, Training Loss: 0.4821, Validation Loss: 0.3688,Validation acc: 84.92
Epoch 3/10, Training Loss: 0.3257, Validation Loss: 0.2908,Validation acc: 86.60
Epoch 4/10, Training Loss: 0.2126, Validation Loss: 0.2489,Validation acc: 87.69
Epoch 5/10, Training Loss: 0.1261, Validation Loss: 0.4647,Validation acc: 87.74
Epoch 6/10, Training Loss: 0.0827, Validation Loss: 0.3894,Validation acc: 88.44
Epoch 7/10, Training Loss: 0.1164, Validation Loss: 0.2903,Validation acc: 88.73
Epoch 8/10, Training Loss: 0.0528, Validation Loss: 0.3281,Validation acc: 88.94
Epoch 9/10, Training Loss: 0.0828, Validation Loss: 0.1929,Validation acc: 89.22
Epoch 10/10, Training Loss: 0.0298, Validation Loss: 0.3574,Validation acc: 89.30


In [9]:
model.load_state_dict(torch.load("best_model.pth"))

<All keys matched successfully>

In [10]:
#Accurcy

model.eval()
correct =0
total = 0

with torch.no_grad():

  for images,labels in test_loader:

    images = images.to(device)
    labels = labels.to(device)

    output = model(images)

    _, predicated = torch.max(output,1)

    total+= labels.size(0)

    correct += (predicated == labels).sum().item()

test_accy = 100*correct / total

print(f"Test Accuracy: {test_accy:.4f}")


Test Accuracy: 88.5000
